# Strict-extractive factoid SFT for Qwen2.5-3B-Instruct

Scale the strict-extractive 0.5B experiment to [Qwen/Qwen2.5-3B-Instruct](https://huggingface.co/Qwen/Qwen2.5-3B-Instruct). Use the same questions, supported-alias targets, full resources, prompt, LoRA settings and checkpoint-selection procedure. The existing trainer uses 4-bit loading and Unsloth gradient checkpointing.

Run the cells in order using the same training environment as the 0.5B notebook. The model-preparation cell downloads missing 3B weights. Training starts in the Train cell. All notebook outputs are cleared.

Prepared data and model outputs use separate `qwen25_3b` directories. Rerunning this notebook resumes its own 3B checkpoints through `--resume-from-checkpoint auto`.


In [ ]:
from __future__ import annotations
import json
import sys
from pathlib import Path

def project_root(start: Path) -> Path:
    for candidate in [start, *start.parents, start / 'Task-Structured Counterfactual Preference Mining']:
        if (candidate / 'src').is_dir() and (candidate / 'cse_dpo').is_dir(): return candidate.resolve()
    raise RuntimeError('Run from the project or workspace directory.')

PROJECT_ROOT=project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from cse_dpo.normalize_set_answers import normalize_answer_surface
from src.utility.bioasq_format import parse_prediction_items
from src.utility.data import list_record_resources
# This directory supplies model-independent question/resource/answer records.
# The trainer renders them again with the 3B tokenizer and selected prompt.
SOURCE_DIR=PROJECT_ROOT/'data/BioASQ_factoid_sft_prepared/single_answer_full_resources_qwen25_05b'
TRAIN_SOURCE=SOURCE_DIR/'train_prepared.json'
DEV_SOURCE=SOURCE_DIR/'eval_prepared.json'
# Choose 'evidence_all_supported' (one row per question), 'evidence_single_supported_alias' (977 singleton questions), or 'evidence_per_supported_alias' (one row per supported alias).
TRAINING_DATA_VARIANT='evidence_per_supported_alias'
#'factoid-single-answer-extractive-v1'
# The literal-copy run keeps all other settings fixed and changes only the factoid instruction.
PROMPT_ID='factoid-single-answer-extractive-v1'
LORA_DROPOUT=0.05
BASE_MODEL_ID='Qwen/Qwen2.5-3B-Instruct'
BASE_MODEL_DIR=PROJECT_ROOT/'models/Qwen2.5-3B-Instruct'
ALL_SUPPORTED_DIR=PROJECT_ROOT/'data/BioASQ_factoid_sft_prepared/evidence_grounded_single_answer_qwen25_3b'
SINGLE_ALIAS_DIR=PROJECT_ROOT/'data/BioASQ_factoid_sft_prepared/evidence_grounded_single_alias_qwen25_3b'
PER_ALIAS_DIR=PROJECT_ROOT/'data/BioASQ_factoid_sft_prepared/evidence_grounded_per_supported_alias_qwen25_3b'
PREPARED_DIR={'evidence_all_supported': ALL_SUPPORTED_DIR, 'evidence_single_supported_alias': SINGLE_ALIAS_DIR, 'evidence_per_supported_alias': PER_ALIAS_DIR}[TRAINING_DATA_VARIANT]
FILTERED_TRAIN=PREPARED_DIR/'train_prepared.json'
DEV_SUPPORTED_IDS=PREPARED_DIR/'dev_supported_question_ids.json'

def supported_aliases(row: dict) -> list[str]:
    evidence=normalize_answer_surface(' '.join(list_record_resources(row)))
    aliases=parse_prediction_items(str(row.get('output','')), 'factoid')
    return [alias for alias in aliases if (normalized:=normalize_answer_surface(alias)) and f' {normalized} ' in f' {evidence} ']

train_rows=json.loads(TRAIN_SOURCE.read_text(encoding='utf-8'))
dev_rows=json.loads(DEV_SOURCE.read_text(encoding='utf-8'))
all_supported_train=[]
single_supported_alias_train=[]
per_supported_alias_train=[]
for row in train_rows:
    aliases=supported_aliases(row)
    if aliases:
        filtered=dict(row); filtered['output']=''.join(f'[BE]{alias}[EE]' for alias in aliases); all_supported_train.append(filtered)
        if len(aliases)==1:
            single_supported_alias_train.append(filtered)
        for alias_index,alias in enumerate(aliases):
            per_alias=dict(row); per_alias['id']=f"{row['id']}__supported_alias_{alias_index}"; per_alias['source_question_id']=row['id']; per_alias['output']=f'[BE]{alias}[EE]'; per_supported_alias_train.append(per_alias)
dev_supported_ids=[row['id'] for row in dev_rows if supported_aliases(row)]
ALL_SUPPORTED_DIR.mkdir(parents=True,exist_ok=True)
SINGLE_ALIAS_DIR.mkdir(parents=True,exist_ok=True)
PER_ALIAS_DIR.mkdir(parents=True,exist_ok=True)
(ALL_SUPPORTED_DIR/'train_prepared.json').write_text(json.dumps(all_supported_train,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
(SINGLE_ALIAS_DIR/'train_prepared.json').write_text(json.dumps(single_supported_alias_train,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
(PER_ALIAS_DIR/'train_prepared.json').write_text(json.dumps(per_supported_alias_train,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
DEV_SUPPORTED_IDS.write_text(json.dumps({'question_ids':dev_supported_ids},indent=2)+'\n',encoding='utf-8')
assert len(all_supported_train)==1130, len(all_supported_train)
assert len(single_supported_alias_train)==977, len(single_supported_alias_train)
assert len(per_supported_alias_train)==1352, len(per_supported_alias_train)
assert all(len(parse_prediction_items(row['output'], 'factoid'))==1 for row in per_supported_alias_train)
assert len(dev_supported_ids)==129, len(dev_supported_ids)
print(f'Training variant: {TRAINING_DATA_VARIANT}')
print(f'LoRA dropout: {LORA_DROPOUT}')
print(f'All evidence-supported train: {len(all_supported_train)} / {len(train_rows)}')
print(f'Single-supported-alias train: {len(single_supported_alias_train)} / {len(train_rows)}')
print(f'Per-supported-alias train: {len(per_supported_alias_train)} rows from {len(all_supported_train)} questions')
selected_train={'evidence_all_supported':all_supported_train, 'evidence_single_supported_alias':single_supported_alias_train, 'evidence_per_supported_alias':per_supported_alias_train}[TRAINING_DATA_VARIANT]
print(f'Selected train: {len(selected_train)}')
print(f'Supported dev: {len(dev_supported_ids)} / {len(dev_rows)}')


## Prepare the 3B base model

Download missing model files from the official Qwen repository. Subsequent training uses local files only.


In [ ]:
import os

# HTTP fallback for the stalled Xet transfer. Reuse completed local files.
os.environ['HF_HUB_DISABLE_XET'] = '1'
from huggingface_hub import snapshot_download, constants as hf_constants
# Also apply when huggingface_hub was already imported by an earlier cell.
hf_constants.HF_HUB_DISABLE_XET = True

# Reuse a complete local download, including all indexed weight shards.
def missing_model_files(model_dir):
    required = {'config.json', 'tokenizer.json', 'tokenizer_config.json'}
    index_path = model_dir / 'model.safetensors.index.json'
    if index_path.exists():
        index = json.loads(index_path.read_text(encoding='utf-8'))
        required.update(index['weight_map'].values())
    else:
        required.add('model.safetensors')
    return sorted(name for name in required if not (model_dir / name).is_file())

missing_before = missing_model_files(BASE_MODEL_DIR)
print('Missing model files:', missing_before, flush=True)
if missing_before:
    print('Downloading via HTTP; completed files are reused. The incomplete shard may restart.', flush=True)
    # Read the credential at runtime; never print it or embed it in notebook source.
    token_path = PROJECT_ROOT.parent / 'hf_token.txt'
    if not token_path.is_file():
        raise FileNotFoundError(f'Missing Hugging Face token file: {token_path}')
    _download_token = token_path.read_text(encoding='utf-8-sig').strip()
    try:
        if not _download_token:
            raise ValueError('The Hugging Face token file is empty.')
        snapshot_download(
            repo_id=BASE_MODEL_ID,
            local_dir=str(BASE_MODEL_DIR),
            token=_download_token,
            max_workers=1,
            allow_patterns=['*.json', '*.safetensors', '*.txt', '*.model', '*.jinja'],
        )
    finally:
        del _download_token
missing = missing_model_files(BASE_MODEL_DIR)
assert not missing, f'Missing 3B model files: {missing}'
model_config = json.loads((BASE_MODEL_DIR / 'config.json').read_text(encoding='utf-8'))
assert model_config.get('model_type') == 'qwen2'
assert model_config.get('num_hidden_layers') == 36, 'Expected the Qwen2.5-3B architecture.'
print('Local base model ready:', BASE_MODEL_DIR)


## Train

Checkpoint selection saves the best all-dev MRR adapter and a separate best evidence-supported MRR adapter. Early stopping treats improvement in either generated MRR, or eval loss, as progress.

Settings match the strict-extractive 0.5B experiment: learning rate `6e-4`, up to 6 epochs, batch size 1 with 32-step accumulation, LoRA rank/alpha 32 and dropout 0.05, training context 4096 and selection context 3584, seed 3407. Keeping the same learning rate provides a controlled starting comparison; it is not a tuned 3B setting.

Use `MODEL_DIR / "adapter_best_evidence_mrr"` to follow the checkpoint choice used for the 0.5B HEA-DPO initialization. A 3B adapter must be evaluated on the 3B base model.


In [ ]:
import subprocess
BASE_MODEL_DIR_NAME={'evidence_all_supported':'evidence_grounded_single_answer_qwen25_3b', 'evidence_single_supported_alias':'evidence_grounded_single_alias_qwen25_3b', 'evidence_per_supported_alias':'evidence_grounded_per_supported_alias_qwen25_3b'}[TRAINING_DATA_VARIANT]
DROPOUT_TAG=f'lora_dropout_{LORA_DROPOUT:.2f}'.replace('.', '')
PROMPT_TAG={
    'factoid-single-answer-aligned-v1': 'aligned',
    'factoid-single-answer-extractive-v1': 'strict_extractive',
    'factoid-single-answer-literal-copy-v1': 'literal_copy',
}[PROMPT_ID]
MODEL_DIR=PROJECT_ROOT/'Artifacts/Factoid_SFT/models'/f'{BASE_MODEL_DIR_NAME}_{DROPOUT_TAG}_{PROMPT_TAG}'
command=[
 sys.executable,'src/utility/answer_gen_ft.py','--train-input',str(FILTERED_TRAIN),'--eval-input',str(DEV_SOURCE),'--question-types','factoid',
 '--prepared-output-dir',str(MODEL_DIR/'trainer_prepared'),'--model-name',str(BASE_MODEL_DIR),
 '--chat-template','qwen-2.5','--prompt-format','chat','--prompt-file','prompts/factoid_single_answer_aligned.json','--prompt',PROMPT_ID,
 '--max-resources','0','--max-resource-chars','0','--max-factoid-answers','5','--max-seq-length','4096',
 '--per-device-train-batch-size','1','--per-device-eval-batch-size','1','--gradient-accumulation-steps','32','--warmup-steps','5','--num-train-epochs','6.0','--learning-rate','6e-4','--weight-decay','0.01',
 '--logging-steps','1','--save-strategy','steps','--save-steps','25','--lora-r','32','--lora-alpha','32','--lora-dropout',str(LORA_DROPOUT),'--dataset-num-proc','1',
 '--selection-metric','generated_primary_score','--generated-metric-subset-ids',str(DEV_SUPPORTED_IDS),'--selection-max-seq-length','3584','--selection-max-new-tokens','64','--selection-num-generations','1','--selection-aggregation-strategy','union',
 '--early-stopping-patience','3','--early-stopping-threshold','0.001','--seed','3407','--run-name',f'bioasq-qwen25-3b-factoid-{TRAINING_DATA_VARIANT}-{PROMPT_TAG}-dropout-{LORA_DROPOUT:.2f}-sft',
 '--output-dir',str(MODEL_DIR/'trainer_output'),'--save-model-dir',str(MODEL_DIR/'adapter'),'--resume-from-checkpoint','auto','--local-files-only']
print(' '.join(command))
subprocess.run(command,cwd=PROJECT_ROOT,check=True)


## Generated Dev Metrics

Each history row contains all-dev MRR on 160 questions and supported-dev MRR on 129 questions. Both come from one decoding pass.


In [ ]:
history_path=MODEL_DIR/'generated_dev_selection/history.json'
best_path=MODEL_DIR/'generated_dev_selection/best_summary.json'
evidence_best_path=MODEL_DIR/'evidence_mrr_selection/best_summary.json'
assert history_path.exists(), f'Missing history: {history_path}'
for entry in json.loads(history_path.read_text(encoding='utf-8')):
    print({'step':entry.get('step'),'epoch':entry.get('epoch'),'all_dev_mrr_160':entry.get('metric_value'),'supported_dev_mrr_129':entry.get('generated_subset_mrr'),'supported_dev_questions':entry.get('generated_subset_question_count')})
if best_path.exists():
    best=json.loads(best_path.read_text(encoding='utf-8'))
    print('Best all-dev checkpoint:',{'step':best.get('step'),'all_dev_mrr_160':best.get('metric_value'),'supported_dev_mrr_129':best.get('generated_subset_mrr')})
if evidence_best_path.exists():
    best=json.loads(evidence_best_path.read_text(encoding='utf-8'))
    print('Best evidence-supported checkpoint:',{'step':best.get('step'),'all_dev_mrr_160':best.get('metric_value'),'supported_dev_mrr_129':best.get('generated_subset_mrr')})
